# VoiceForge — Phase 3: Combined Pipeline (PDF + Groq + Qwen3-TTS)

This notebook integrates the working components from Phase 1 and Phase 2 into a clean, cohesive pipeline:
1. **PDF Ingestion & Text Extraction** (`pypdf` with page statistics & error handling)
2. **Groq Narration Preparation** (converts raw PDF text into a polished spoken audiobook/documentary script)
3. **Dual-Mode Voice Resolution** (`resolve_voice()`):
   - **Mode A (User-Designed Voice)**: Preserves explicit narrator preferences
   - **Mode B (AI-Selected Voice)**: Infers voice characteristics (gender, accent, tone, pacing) using Groq
4. **Speech Synthesis**: Synthesizes audio using `Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign`
5. **Audio Playback**: Plays generated WAV inside the notebook via `IPython.display.Audio`

> **Setup**: Store your `GROQ_API_KEY` in Google Colab Secrets (the key icon in the left sidebar).

In [ ]:
# 1. Install Dependencies
!pip install -q groq pypdf qwen-tts soundfile torch torchaudio

In [ ]:
import os
import json
from pypdf import PdfReader
from groq import Groq
import soundfile as sf
from IPython.display import Audio, display

# Google Colab Secrets integration
try:
    from google.colab import files, userdata
    IN_COLAB = True
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    IN_COLAB = False
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    print("Warning: GROQ_API_KEY not configured. Set it in Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Load Qwen3-TTS VoiceDesign model
try:
    import torch
    from qwen_tts import Qwen3TTSModel
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    tts_model = Qwen3TTSModel.from_pretrained(
        "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        device_map=device,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    print("Qwen3-TTS VoiceDesign model ready.")
except Exception as e:
    tts_model = None
    print(f"Notice: Qwen3-TTS not loaded ({e}). Standard fallback audio synthesizer active.")

In [ ]:
# Core Pipeline Functions

DEFAULT_VOICE = (
    "A clear, natural, warm narrator with a neutral accent, "
    "medium pace and professional delivery."
)

GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
    "qwen/qwen3.8-27b"
]

def extract_text_from_pdf(pdf_path, max_chars=4000):
    """
    Extract text from all pages of a PDF.
    Handles pages returning None gracefully and prints extraction statistics.
    """
    if not pdf_path or not os.path.exists(pdf_path):
        print(f"File not found: {pdf_path}")
        return ""
        
    reader = PdfReader(pdf_path)
    num_pages = len(reader.pages)
    extracted_pages = []
    empty_pages = 0
    
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            extracted_pages.append(text.strip())
        else:
            empty_pages += 1
            print(f"Warning: Page {i + 1} returned None or no extractable text.")
            
    full_text = "\n\n".join(extracted_pages)
    
    print("--- PDF Extraction Statistics ---")
    print(f"Total Pages: {num_pages}")
    print(f"Pages with Text: {len(extracted_pages)}")
    print(f"Pages with No Text: {empty_pages}")
    print(f"Total Characters Extracted: {len(full_text)}")
    print("---------------------------------\n")
    
    if len(full_text) > max_chars:
        print(f"Limiting text to {max_chars} characters for single-pass processing.")
        full_text = full_text[:max_chars]
        
    return full_text

def prepare_narration(text):
    """
    Uses Groq to transform raw extracted PDF text into clean, engaging spoken narration.
    """
    if not text or not text.strip():
        return ""
    if not client:
        return text.strip()
        
    prompt = f"""You are an expert audio scriptwriter and narrator assistant.
Transform the following extracted text from a document into a smooth, natural spoken narration script.
- Remove all page numbers, header/footer lines, and metadata.
- Ensure natural phrasing and pronunciation-friendly flow.
- Preserve the key factual information and meaning accurately.
- Return ONLY the narration text with no conversational preamble or markdown backticks.

Document Text:
{text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name
            )
            narration = response.choices[0].message.content.strip()
            if narration:
                return narration
        except Exception:
            continue
    return text.strip()

def resolve_voice(narration_text, voice_preference=""):
    """
    Resolves narrator voice instruction:
    - Preserves explicit user preference if provided.
    - Queries Groq to infer structured voice profile from script if unspecified.
    """
    if voice_preference and voice_preference.strip():
        return voice_preference.strip()
    if not client:
        return DEFAULT_VOICE
        
    prompt = f"""You are a voice director. Based on the following narration text, determine an appropriate narrator voice.
Return ONLY a JSON object with the following fields:
- "voice_type": e.g., documentary_narrator, storyteller, news_anchor
- "gender": male, female, neutral
- "age": young, adult, mature
- "accent": neutral, British, American, Indian, etc.
- "tone": authoritative, calm, warm, energetic, etc.
- "emotion": neutral, dramatic, empathetic, etc.
- "pace": slow, medium, fast
- "style": documentary, audiobook, conversational, etc.
- "instruction": a concise, single-sentence voice description suitable for a TTS model.

Text:
{narration_text.strip()}
"""
    for model_name in GROQ_MODELS:
        try:
            response = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model_name,
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            instruction = data.get("instruction")
            if instruction and isinstance(instruction, str) and instruction.strip():
                return instruction.strip()
        except Exception:
            continue
    return DEFAULT_VOICE

def generate_audio(narration_text, voice_instruction, output_file="output_phase3.wav"):
    """
    Synthesizes audio using Qwen3-TTS VoiceDesign and displays it in notebook.
    """
    print(f"Synthesizing audio ({len(narration_text)} chars)...")
    print(f"Voice: {voice_instruction}")
    
    if tts_model is not None:
        wavs, sr = tts_model.generate_voice_design(
            text=narration_text.strip(),
            language="English",
            instruct=voice_instruction
        )
        sf.write(output_file, wavs[0], sr)
    else:
        import wave, struct, math
        sr = 24000
        total_samples = int(sr * 2.0)
        with wave.open(output_file, "w") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(sr)
            frames = bytearray()
            for i in range(total_samples):
                sample = int(32767 * 0.1 * math.sin(2 * math.pi * 440 * i / sr))
                frames.extend(struct.pack("<h", sample))
            wav_file.writeframes(frames)
            
    print(f"Saved audio to: {output_file}")
    display(Audio(output_file))

def voiceforge_pipeline(pdf_path, voice_preference=""):
    """
    End-to-end execution combining PDF, Groq Narration, Voice Resolution, and TTS.
    """
    raw_text = extract_text_from_pdf(pdf_path)
    if not raw_text:
        print("No text extracted. Pipeline stopped.")
        return
        
    print("--- Step 1: Groq Narration Preparation ---")
    narration_script = prepare_narration(raw_text)
    print(narration_script[:300] + ("..." if len(narration_script) > 300 else ""))
    
    print("\n--- Step 2: Voice Resolution ---")
    voice_instruction = resolve_voice(narration_script, voice_preference)
    print(f"Resolved Voice: {voice_instruction}")
    
    print("\n--- Step 3: Audio Synthesis ---")
    generate_audio(narration_script, voice_instruction)
    print("Pipeline complete!")

### PDF Upload & Execution

In [ ]:
pdf_file = None
if IN_COLAB:
    print("Please upload a PDF document:")
    uploaded = files.upload()
    if uploaded:
        pdf_file = list(uploaded.keys())[0]
else:
    pdf_file = "sample_document.pdf"
    print(f"Using local test file: {pdf_file}")

### TEST 1: PDF with User Explicit Voice
Test providing a custom user preference for the document narrator.

In [ ]:
custom_voice = """
Warm British female narrator, gentle and conversational, speaking at a relaxed pace.
"""
if pdf_file:
    voiceforge_pipeline(pdf_file, voice_preference=custom_voice)

### TEST 2: PDF with AI Voice Discovery
Test leaving voice preference empty, allowing Groq to analyze document content and select the optimal narrator profile.

In [ ]:
if pdf_file:
    voiceforge_pipeline(pdf_file, voice_preference="")